# Module 5 — Compound Agent (Genie + KA + MCP + ai_query)

**Time**: ~30 minutes

**For analysts**: We give one agent four tools — Genie for SQL, the Knowledge Assistant for documents, a Fetch tool for live web pages, and a parser for ad-hoc PDFs — and let it decide which to call. One question, multiple tools, one answer.

**For engineers**: We're building a `ResponsesAgent` (Mosaic AI's recommended agent abstraction), registering 4 tools, logging it with MLflow, and deploying to Model Serving as `disa-cti-agent`.

## The four tools

| Tool | Backed by | Use when |
|---|---|---|
| `genie_query` | Module 3 Genie space | Question is structured / quantitative |
| `knowledge_assistant_search` | Module 4 KA endpoint | Question is doctrinal / cites a control |
| `fetch_advisory_url` | Fetch MCP (free, built-in) | User pasted a cisa.gov URL |
| `parse_pdf` | `ai_parse_document` | User uploaded a PDF |

In [ ]:
%pip install -U databricks-agents mlflow databricks-sdk
dbutils.library.restartPython()

## 1. Define the agent and its tools

In [ ]:
%%writefile agent.py
"""DISA CTI compound agent.

Tools:
  - genie_query: ask the DISA Threat Intel Genie space
  - knowledge_assistant_search: ask the KA over advisories + STIGs
  - fetch_advisory_url: fetch a public URL (CISA advisory page) via the Fetch MCP server
  - parse_pdf: parse a PDF in a UC volume via ai_parse_document
"""
from __future__ import annotations

import json
import os
import re
from typing import Any, Generator

import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse, ResponsesAgentStreamEvent

GENIE_SPACE_ID = os.getenv("GENIE_SPACE_ID", "")
KA_ENDPOINT = os.getenv("KA_ENDPOINT", "agents_disa-cti-knowledge")
LLM_ENDPOINT = os.getenv("LLM_ENDPOINT", "databricks-claude-sonnet-4-6")

SYSTEM_PROMPT = """You are the DISA Cyber Threat Intelligence assistant.

You have four tools:
  - genie_query(question): asks structured questions over CVE / KEV / STIG / asset tables. Use for counts, joins, filters, time series.
  - knowledge_assistant_search(question): retrieves passages from CISA advisories, DoD STIGs, and MITRE ATT&CK descriptions. Use for doctrinal or definitional questions.
  - fetch_advisory_url(url): fetches a public web page and returns its text. Use when the user pastes a cisa.gov URL.
  - parse_pdf(volume_path): parses a PDF in a UC volume. Use when the user references a file path.

Plan: call the most relevant tool first, then chain calls if needed (e.g., fetch advisory -> extract CVEs -> genie_query for asset exposure).
Cite CVE IDs, STIG IDs, and ATT&CK technique IDs in your answers. Keep responses focused and analyst-actionable.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "genie_query",
            "description": "Ask a natural-language question over CVE/KEV/STIG/asset tables.",
            "parameters": {
                "type": "object",
                "properties": {"question": {"type": "string"}},
                "required": ["question"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "knowledge_assistant_search",
            "description": "Retrieve passages from CISA advisories, DoD STIGs, and ATT&CK technique pages.",
            "parameters": {
                "type": "object",
                "properties": {"question": {"type": "string"}},
                "required": ["question"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_advisory_url",
            "description": "Fetch a public web URL and return its text content. Use for live cisa.gov advisories.",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string"}},
                "required": ["url"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "parse_pdf",
            "description": "Parse a PDF in a UC volume using ai_parse_document.",
            "parameters": {
                "type": "object",
                "properties": {"volume_path": {"type": "string"}},
                "required": ["volume_path"],
            },
        },
    },
]


class CTIAgent(ResponsesAgent):
    def __init__(self) -> None:
        self.w = WorkspaceClient()

    def _genie_query(self, question: str) -> str:
        if not GENIE_SPACE_ID:
            return "Genie space not configured."
        conv = self.w.genie.start_conversation_and_wait(space_id=GENIE_SPACE_ID, content=question)
        attachments = conv.attachments or []
        if not attachments:
            return conv.content or "No response."
        result = self.w.genie.get_message_attachment_query_result(
            space_id=GENIE_SPACE_ID,
            conversation_id=conv.conversation_id,
            message_id=conv.message_id,
            attachment_id=attachments[0].attachment_id,
        )
        return json.dumps(result.statement_response.result.data_array[:20])

    def _ka_search(self, question: str) -> str:
        resp = self.w.serving_endpoints.query(
            name=KA_ENDPOINT, messages=[{"role": "user", "content": question}]
        )
        return resp.choices[0].message.content

    def _fetch_url(self, url: str) -> str:
        # In production we would call the Fetch MCP server registered as a UC HTTP connection.
        # For workshop simplicity we fall back to a direct requests.get with a user-agent.
        import requests
        if not url.startswith(("https://www.cisa.gov", "https://nvd.nist.gov", "https://attack.mitre.org")):
            return f"Refusing to fetch {url}: only CISA / NVD / MITRE domains allowed."
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "DISA-Workshop"})
            text = re.sub(r"<[^>]+>", " ", r.text)
            text = re.sub(r"\s+", " ", text).strip()
            return text[:8000]
        except Exception as e:
            return f"Fetch failed: {e}"

    def _parse_pdf(self, volume_path: str) -> str:
        if not volume_path.startswith("/Volumes/disa_workshop/"):
            return "Refusing to parse: path must be under /Volumes/disa_workshop/."
        spark = self.w._spark if hasattr(self.w, "_spark") else None  # placeholder
        try:
            from pyspark.sql import SparkSession
            spark = SparkSession.builder.getOrCreate()
            df = spark.sql(
                f"SELECT ai_parse_document(content).document.pages[0].content AS preview "
                f"FROM READ_FILES('{volume_path}', format => 'binaryFile')"
            )
            row = df.collect()[0]
            return row.preview[:6000]
        except Exception as e:
            return f"Parse failed: {e}"

    def _dispatch(self, name: str, args: dict[str, Any]) -> str:
        if name == "genie_query":
            return self._genie_query(args["question"])
        if name == "knowledge_assistant_search":
            return self._ka_search(args["question"])
        if name == "fetch_advisory_url":
            return self._fetch_url(args["url"])
        if name == "parse_pdf":
            return self._parse_pdf(args["volume_path"])
        return f"Unknown tool {name}"

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for m in request.input:
            messages.append({"role": m.role, "content": m.content})

        for _ in range(6):
            resp = self.w.serving_endpoints.query(
                name=LLM_ENDPOINT, messages=messages, tools=TOOLS
            )
            choice = resp.choices[0].message
            tool_calls = getattr(choice, "tool_calls", None) or []
            if not tool_calls:
                return ResponsesAgentResponse(
                    output=[{"role": "assistant", "content": choice.content or ""}]
                )
            messages.append({"role": "assistant", "content": choice.content or "", "tool_calls": tool_calls})
            for tc in tool_calls:
                args = json.loads(tc.function.arguments)
                result = self._dispatch(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result[:6000]})

        return ResponsesAgentResponse(
            output=[{"role": "assistant", "content": "Hit max tool-call iterations."}]
        )


mlflow.models.set_model(CTIAgent())


## 2. Smoke test the agent locally

In [ ]:
import os
os.environ["GENIE_SPACE_ID"] = "<paste-your-space-id-here>"
os.environ["KA_ENDPOINT"] = "agents_disa-cti-knowledge"
os.environ["LLM_ENDPOINT"] = "databricks-claude-sonnet-4-6"

from agent import CTIAgent
from mlflow.types.responses import ResponsesAgentRequest

agent = CTIAgent()
req = ResponsesAgentRequest(input=[{"role": "user", "content": "How many KEV-listed CVEs were added in the last 30 days, and which vendors do they target?"}])
resp = agent.predict(req)
print(resp.output[-1].content)

## 3. Log + register the agent

In [ ]:
import mlflow
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="disa-cti-agent"):
    info = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",
        registered_model_name="disa_workshop.threat_intel.disa_cti_agent",
        pip_requirements=["mlflow", "databricks-sdk", "databricks-agents", "requests", "pyspark"],
        resources=[
            DatabricksServingEndpoint(endpoint_name="databricks-claude-sonnet-4-6"),
            DatabricksServingEndpoint(endpoint_name="agents_disa-cti-knowledge"),
            DatabricksFunction(function_name="disa_workshop.threat_intel.lookup_cve"),
            DatabricksFunction(function_name="disa_workshop.threat_intel.assets_for_product"),
        ],
    )
print(f"Logged model: {info.model_uri}")

## 4. Deploy to Model Serving

In [ ]:
from databricks.agents import deploy

deployment = deploy(
    model_name="disa_workshop.threat_intel.disa_cti_agent",
    model_version=info.registered_model_version,
    endpoint_name="disa-cti-agent",
    environment_vars={
        "GENIE_SPACE_ID": os.environ["GENIE_SPACE_ID"],
        "KA_ENDPOINT": os.environ["KA_ENDPOINT"],
        "LLM_ENDPOINT": os.environ["LLM_ENDPOINT"],
    },
)
print(f"Deployment: {deployment}")

## 5. End-to-end test

Hit the endpoint with a question that exercises **all four tools**.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

test_question = (
    "Fetch https://www.cisa.gov/news-events/cybersecurity-advisories "
    "and tell me which CVEs are mentioned. Then check our asset inventory "
    "for any exposure, and tell me which STIG controls would mitigate."
)

resp = w.serving_endpoints.query(
    name="disa-cti-agent",
    messages=[{"role": "user", "content": test_question}],
)
print(resp.choices[0].message.content)

**Next**: **Module 6** — open `app/` in your editor, point `app.yaml`'s `serving_endpoint` at `disa-cti-agent`, and run `npm run dev`. Then we live-vibe-code a new chart using the prompt at `prompts/vibe_code_component.md`.